# Proyecto Final: Análisis de la Industria de Videojuegos con Datos de IGDB

Este cuaderno consolida el proceso completo de Ciencia de Datos, abarcando desde la extracción de datos (ETL) hasta el Análisis Exploratorio (EDA) y la definición de hipótesis para el modelado predictivo.

---

## 1. Fase de Extracción de Datos (Extraction)

### 1.1. Estrategia de Recolección de Datos
Para este proyecto, hemos optado por una **Fuente de Datos Secundaria y Externa**. En lugar de utilizar archivos estáticos o realizar *Web Scraping* (que puede ser inestable y éticamente cuestionable por la carga al servidor), utilizamos la **API oficial de IGDB**.

**Justificación de la elección:**
* **Estructura:** La API nos entrega datos en formato **JSON (semi-estructurado)**. Como vimos en la clase de "Tipos de Datos", esto nos ofrece flexibilidad en la ingesta, aunque requiere un pre-procesamiento posterior para tabularlo en DataFrames.
* **Acceso:** Utilizamos la librería `requests` para gestionar las peticiones `POST`, implementando el protocolo de autenticación `Client-ID` y `Authorization` requerido.

### 1.2. Paginación y Respeto a los Límites (Rate Limiting)
Dado el volumen de datos, no es posible realizar una única petición (el *payload* sería excesivo y la conexión podría cortarse). Implementamos una función de **descarga iterativa (paginada)** usando los parámetros `offset` y `limit`.

**Decisión Técnica Ética:**
Se incluyó un `time.sleep(0.4)` entre cada iteración.
* **Por qué:** Para respetar los límites de velocidad de la API y evitar bloqueos o caídas del servicio. Esto garantiza una extracción de datos robusta y profesional, alineada con las buenas prácticas de recolección de datos en la web.

---

## 2. Fase de Limpieza y Transformación (Wrangling)

### 2.1. Introducción al Proceso de Data Wrangling
Una vez extraídos los datos crudos (`raw`), nos encontramos con problemas típicos mencionados en la clase de **Transformación**: datos faltantes, tipos de datos incorrectos (listas guardadas como strings) y registros "ruido". Este proceso documenta el paso de datos crudos a datos procesados (`processed`), listos para el análisis.

### 2.2. Transformación de Estructuras Complejas
**Problema:** Al guardar los datos en CSV (formato tabular/plano), las columnas que originalmente eran listas en el JSON (como `genres` o `platforms`) se convirtieron en cadenas de texto (ej: `"[1, 2, 3]"`). Pandas las interpreta como `object` (string).

**Solución:** Utilizamos `literal_eval` de la librería `ast`.

**Justificación:** A diferencia de un simple *split* de strings, `literal_eval` evalúa la cadena con seguridad y reconstruye la estructura de lista original de Python. Esto es crucial para, posteriormente, poder "desenrollar" o explotar estas relaciones (One-to-Many) en el análisis.

### 2.3. Ventana Temporal y Calidad del Dato
**Decisión:** Filtramos juegos lanzados entre el año 2000 y 2025.

**Justificación:**
* **Consistencia:** Los datos de juegos muy antiguos suelen tener muchos valores nulos en metadatos modernos (como ratings detallados), lo que introduciría sesgo en el análisis.
* **Relevancia:** Nos enfocamos en la era moderna de la industria para que las comparaciones de plataformas y géneros sean coherentes con el mercado actual.

### 2.4. Tratamiento de Duplicados (Lógica de Negocio)
Detectamos que un mismo juego puede aparecer múltiples veces (por ejemplo, re-lanzamientos, ediciones GOTY, etc.).

**Estrategia:** En lugar de eliminar arbitrariamente (ej. `drop_duplicates` simple), aplicamos una **eliminación lógica**:
1.  Ordenamos por `total_rating_count` descendente.
2.  Mantenemos la entrada con mayor cantidad de votos.

**Por qué:** Asumimos que la entrada con más interacción de usuarios es la "versión principal" o la más representativa del juego, preservando la información de mayor calidad (Clase "Limpieza de Datos").

### 2.5. Integridad Referencial y Normalización (Tablas Auxiliares)
Además de la tabla principal de juegos, contamos con tablas auxiliares (`generos`, `plataformas`, `empresas`).

**Justificación:**
Estamos trabajando con un modelo relacional. En lugar de tener una sola tabla gigante con texto repetido (ej: repetir la palabra "Adventure" mil veces), mantenemos tablas de dimensiones enlazadas por `ID`.
1.  Los `ID` se aseguran como tipo correcto (`Int64`) para permitir cruces (`merges`) limpios.
2.  Solo conservamos registros auxiliares que tengan relación con los juegos filtrados en el paso anterior, reduciendo el tamaño de los datos y eliminando basura.

---

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1. Introducción y Objetivos
Siguiendo la metodología vista en el curso, el EDA no es un paso rígido, sino una "actitud de flexibilidad". Nuestros objetivos en esta etapa son:
1.  **Validar la calidad de los datos:** Verificamos si tenemos suficientes datos de *rating* para responder nuestras preguntas.
2.  **Análisis Univariado:** Entender las distribuciones de nuestras variables principales (Años, Ratings, Reseñas).
3.  **Análisis Bivariado y Multivariado:** Buscar relaciones entre el tipo de estudio, la plataforma y el éxito del juego.

Este paso es crucial para decidir qué **Algoritmos de Aprendizaje Supervisado** (Clasificación) podremos usar en la siguiente etapa.

### 3.2. Análisis de Datos Faltantes (Profiling)
Es vital entender *por qué* faltan datos. En el caso de los videojuegos, la falta de `total_rating` o `total_rating_count` no suele ser aleatoria (MCAR), sino **No Aleatoria (MNAR)**: los juegos menos populares o de nicho simplemente no reciben reseñas.

**Decisión:**
Analizaremos qué porcentaje de los datos está completo. Para el modelado predictivo del "éxito", trabajaremos exclusivamente con el subconjunto de datos que posee métricas de evaluación, asumiendo que *no tener datos* equivale a una falta de tracción en el mercado.

### 3.3. Ingeniería de Características: Variable Objetivo (Target)
Para preparar el terreno hacia un problema de **Aprendizaje Supervisado de Clasificación**, necesitamos definir una variable dependiente ($Y$) clara.

Creamos la variable binaria `exitoso` ($1$ o $0$) basada en dos umbrales:
* **Calidad:** `total_rating` $\ge$ 75 (Un juego bien recibido).
* **Popularidad:** `total_rating_count` $\ge$ 10 (Evita sesgos de juegos con una sola reseña perfecta).

Esto convierte nuestro problema de predecir un número (Regresión) en uno de predecir una clase (Clasificación), lo cual suele ser más robusto para datos con mucho ruido.

### 3.4. Análisis Bivariado: El impacto del presupuesto (AAA vs Indie)
Utilizaremos **Boxplots (Diagramas de Caja)** para comparar las distribuciones de ratings y reseñas entre estudios AAA y No AAA.

**Justificación:**
Como vimos en la clase de Visualización, el *Boxplot* es superior al promedio simple porque nos muestra la **Mediana** (menos sensible a valores extremos) y los **Outliers** (valores atípicos), que en la industria de videojuegos (grandes éxitos virales) son fundamentales.

---

## 4. Conclusiones del EDA y Respuestas a las Preguntas de Investigación

Tras explorar las distribuciones y relaciones en los datos, podemos responder a las preguntas planteadas al inicio del proyecto con evidencia estadística visual.

### 4.1. ¿Qué géneros dominan el mercado? (Cantidad vs. Calidad)
* **Volumen:** El género **"Indie"** satura el mercado con más de 78,000 títulos, lo que indica una barrera de entrada baja.
* **Éxito Real:** A pesar del volumen, el mapa de calor revela que la tasa de éxito de los "Indie" genéricos es baja. En contraste, géneros de nicho complejo como **RPG y Estrategia** muestran una intensidad de éxito consistente.
* **Insight:** La popularidad de desarrollo no equivale a popularidad de recepción. Los nichos complejos tienen bases de jugadores más fieles y críticas más sólidas.

### 4.2. La Brecha AAA vs. Indie
* **Probabilidad de Éxito:** Existe un abismo estadístico. Un juego AAA tiene un **36.6%** de probabilidad de éxito, mientras que un Indie sin publisher cae al **2.7%**.
* **El Factor Visibilidad:** Los gráficos de caja (Boxplots) revelaron que, aunque la calidad (Rating) puede ser similar entre un Indie y un AAA, la **cantidad de reseñas** es inmensamente superior en los AAA.
* **Conclusión:** El músculo de marketing y distribución es determinante. Sin visibilidad (reseñas), no hay éxito en nuestro modelo, independientemente de la calidad técnica.

### 4.3. Estilos de Juego y Comunidad
* Cruzando los datos de géneros, inferimos que los juegos de ciclos cortos (Arcade/Casual) tienen dificultades para acumular las 10 reseñas necesarias para ser considerados "exitosos".
* Los juegos profundos (Simuladores, RPGs) fomentan comunidades que discuten y reseñan, otorgando **visibilidad orgánica** a largo plazo.

### 4.4. Plataformas: ¿Dónde debe lanzar un Indie?
* Al filtrar exclusivamente por juegos *No AAA*, los datos sugieren que **PC y Consolas** ofrecen mayor probabilidad de éxito que los móviles.
* **Por qué:** Las tiendas de aplicaciones móviles sufren graves problemas de descubribilidad por saturación. En PC/Consola, el público está más predispuesto a interactuar y calificar el producto.

### 4.5. El Mito de la Estacionalidad (Ventanas de Lanzamiento)
* **El Mito:** "Evitar el final de año por la competencia con los gigantes".
* **Los Datos:**
    * **Enero (8.4% éxito):** El peor mes. Posible "resaca de gastos" post-navidad.
    * **Septiembre/Octubre (~13% éxito):** Los picos más altos.
* **Estrategia:** Existe una ventana de atención clara en el tercer trimestre (Q3), coincidiendo con el regreso a la rutina, que los desarrolladores independientes están aprovechando mejor que el inicio del año.

---

### Siguiente Paso: Modelado Predictivo
Con estos *insights*, definimos nuestro objetivo para la siguiente etapa: **Entrenar un modelo de clasificación** que ayude a un desarrollador a estimar su probabilidad de éxito $P(Y=1|X)$ basándose en sus decisiones de diseño (Género, Plataforma) y estrategia de lanzamiento (Mes, Publisher), antes de escribir una sola línea de código del juego.

# 5. Conclusiones del EDA y Respuestas a las Preguntas de Investigación

A continuación, presentamos los hallazgos visuales y estadísticos que responden a las preguntas planteadas al inicio del proyecto.

---

### 1. ¿Qué géneros dominan el mercado? (Cantidad vs. Calidad)

Al cruzar el año de lanzamiento con el género y la tasa de éxito, obtenemos el siguiente mapa de calor:

![Mapa de Calor: Éxito por Género y Año](img/mapa_calorr.png)

* **Volumen vs. Éxito:** Aunque el género **"Indie"** satura el mercado en cantidad, el mapa revela que su tasa de éxito (colores claros) es baja comparada con otros.
* **Nichos Fuertes:** Géneros de mayor complejidad como **RPG (Role-playing)** y **Strategy** muestran colores más intensos consistentemente. Esto sugiere que, aunque hay menos juegos, tienen bases de jugadores más fieles y críticas más sólidas.

---

### 2. La Brecha AAA vs. Indie

Para entender las diferencias estructurales, comparamos las distribuciones de ratings y la cantidad de reseñas (visibilidad) según el tipo de estudio:

![Boxplots: Comparación AAA vs No AAA](img/aaa_noaaa.png)

* **Probabilidad de Éxito:** Existe un abismo estadístico. Un juego AAA tiene un **36.6%** de probabilidad de éxito, mientras que un Indie sin publisher cae al **2.7%**.
* **El Factor Visibilidad:** Como se observa en el gráfico de la derecha (escala logarítmica), la mediana de reseñas para un juego AAA es inmensamente superior.
* **Conclusión:** El músculo de marketing es determinante. Sin visibilidad (reseñas), no hay éxito en nuestro modelo, incluso si el juego tiene buena nota técnica (gráfico izquierdo).

---

### 3. Estilos de Juego y Comunidad

![Gráfico de Barras: Éxito por Género](.png)

* Observamos que los juegos con ciclos de vida cortos (Arcade/Casual) tienen dificultades para acumular las 10 reseñas necesarias.
* Los juegos profundos fomentan comunidades que discuten y reseñan, otorgando **visibilidad orgánica** a largo plazo.

---

### 4. Plataformas: ¿Dónde debe lanzar un Indie?

Al filtrar exclusivamente por juegos *No AAA*, analizamos qué plataformas ofrecen mejor tasa de éxito:

![Gráfico de Barras: Éxito por Plataforma](img/exito_plat.png)

* **PC y Consolas:** Los datos sugieren que estas plataformas ofrecen mayor probabilidad de éxito que los móviles.
* **Por qué:** Las tiendas de aplicaciones móviles sufren graves problemas de descubribilidad. En PC/Consola, el público está más predispuesto a interactuar y calificar el producto.

---

### 5. El Mito de la Estacionalidad (Ventanas de Lanzamiento)

Analizamos la proporción de éxito según el mes de lanzamiento para validar si existen "mejores momentos" para publicar:

![Gráfico de Barras: Éxito por Mes](img/exito_mes.png)

* **Enero (8.4% éxito):** El peor mes, posiblemente debido a la "resaca de gastos" post-navidad.
* **Septiembre/Octubre (~13% éxito):** Los picos más altos de éxito para indies.
* **Estrategia:** Existe una ventana de atención clara en el tercer trimestre (Q3), coincidiendo con el regreso a la rutina, que los desarrolladores independientes están aprovechando mejor que el inicio del año.

---

### Siguiente Paso: Modelado Predictivo

Con estos *insights*, definimos nuestro objetivo final: **Entrenar un modelo de clasificación** que estime la probabilidad de éxito $P(Y=1|X)$ basándose en estas variables clave (Género, Plataforma, Mes, Publisher) antes del desarrollo.